Copyright © 2024, Andrea Mastropietro. All rights reserved.

This code is licensed under the MIT License.

See the LICENSE file in the project root for more information.

## Important neighbor atom analysis

### Import Libraries

In [ ]:
# Set up proxy and system path
import os
# os.environ["http_proxy"] = "http://web-proxy.informatik.uni-bonn.de:3128"
# os.environ["https_proxy"] = "http://web-proxy.informatik.uni-bonn.de:3128"

import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Standard libraries
import copy
import random
import pickle
import yaml

# Scientific computing
import numpy as np
import torch

# Progress bar
from tqdm.auto import tqdm

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Chemistry and molecular graph
from pysmiles import read_smiles

# Project-specific modules
from src.utils import (
    visualize_mapping_graph,
    visualize_mapping_structure,
    save_xyz_file,
    compute_hausdorff_distance_batch
)
from src.difflinker.lightning import DDPM
from src.difflinker.datasets import get_dataloader


In [2]:
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# Load configuration from config.yml
with open('../config.yml', 'r') as file:
    config = yaml.safe_load(file)

checkpoint = "../" + config['CHECKPOINT']
SAVE_FOLDER = "../results/neighbor_atom_analysis"
DATA_FOLDER = "../" + config['DATA_FOLDER']
DATASET_NAME = config['DATASET_NAME']
keep_frames = int(config['KEEP_FRAMES'])
P = config['P']
device = config['DEVICE'] if torch.cuda.is_available() else 'cpu'
SEED = int(config['SEED'])
ROTATE = config['ROTATE']
TRANSLATE = config['TRANSLATE']
REFLECT = config['REFLECT']
TRANSFORMATION_SEED = int(config['TRANSFORMATION_SEED'])
SAVE_VISUALIZATION = config['SAVE_VISUALIZATION']
M = int(config['M'])
NUM_SAMPLES = int(config['NUM_SAMPLES'])
PARALLEL_STEPS = int(config['PARALLEL_STEPS'])
LOAD_INITIAL_DISTRIBUTION = config['LOAD_INITIAL_DISTRIBUTION']
ATOM_INJECTION = config['ATOM_INJECTION']

INTIAL_DISTIBUTION_PATH = "../datasets/initial_distributions/seed_" + str(SEED)
SHAPLEY_VALUE_FOLDER = "../results/explanations/zinc_final_test/explanations_seed_42/shapley_values"

print("Random seed: ", SEED)

transformations = []
if ROTATE:
    transformations.append("rotate")
if TRANSLATE:
    transformations.append("translate")
if REFLECT:
    transformations.append("reflect")

transformations_str = "_".join(transformations) if transformations else ""

if transformations:
    mapping_output_dir = os.path.join(SAVE_FOLDER, DATASET_NAME, f'seed_{SEED}_{transformations_str}')
else:
    mapping_output_dir = os.path.join(SAVE_FOLDER, DATASET_NAME, f'seed_{SEED}')

    
os.makedirs(mapping_output_dir, exist_ok=True)

# Loading model form checkpoint 
model = DDPM.load_from_checkpoint(checkpoint, map_location=device)

# Possibility to evaluate on different datasets (e.g., on CASF instead of ZINC)
model.val_data_prefix = DATASET_NAME

print(f"Running on device: {device}")

model.data_path = DATA_FOLDER

model = model.eval().to(device)
model.setup(stage='val')
dataloader = get_dataloader(
    model.val_dataset,
    batch_size=1,
)

diffusion_steps = model.edm.T
injection_step = diffusion_steps // 10

Random seed:  42


/home/mastropietro/anaconda3/envs/diff_explainer/lib/python3.10/site-packages/lightning_fabric/utilities/cloud_io.py:57: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
Lightning automatically

Running on device: cuda:0


### Set random seeds

In [3]:
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
np.random.seed(SEED)
random.seed(SEED)

### Important atom analysis

In [4]:
num_samples = NUM_SAMPLES
sampled = 0
start = 0

data_list = []
for data in dataloader:

    if sampled < num_samples:
        data_list.append(data)
        sampled += 1


# load initial distrubution of noisy features and positions
noisy_features = torch.load(INTIAL_DISTIBUTION_PATH + "/noisy_features_seed_" + str(SEED) + ".pt", map_location=device, weights_only=True)
noisy_positions = torch.load(INTIAL_DISTIBUTION_PATH + "/noisy_positions_seed_" + str(SEED) + ".pt", map_location=device, weights_only=True)

### Neighbor atom removal
Choosing exemplary case studies

In [ ]:
top_k_neighs_dict = {3:5, 4:3, 7:8, 10: 6, 13: 10, 14:5,  22: 5, 23: 9, 27: 10, 28:7, 29: 8}
num_perturbations = 1

for data_index, data in enumerate(tqdm(data_list)):

    if data_index not in top_k_neighs_dict.keys():
        continue

    top_k_neighs = top_k_neighs_dict[data_index]

    smile = data["name"][0]
    mol = read_smiles(smile)

    noisy_positions_present_atoms = noisy_positions.clone()
    noisy_features_present_atoms = noisy_features.clone()

    noisy_positions_present_atoms = noisy_positions_present_atoms[:, :data["positions"].shape[1], :]
    noisy_features_present_atoms = noisy_features_present_atoms[:, :data["one_hot"].shape[1], :]

    num_fragment_atoms = int(data["fragment_mask"].sum().item())
    
    chain_batch_original, node_mask_original = model.sample_chain(data, keep_frames=keep_frames, noisy_positions=noisy_positions_present_atoms, noisy_features=noisy_features_present_atoms)
    
    #load Shapley values for Hausdorff distance
    phi_values = []
    
    
    with open(f"{SHAPLEY_VALUE_FOLDER}/shapley_values_atoms_{data_index}.txt", "r") as read_file:
        read_file.readline()
        read_file.readline()
        for row in read_file:
            if row.strip() == "":
                break
            line = row.strip().split(",")
            phi_values.append(float(line[1])) 


    chain_batch_original_molecule = chain_batch_original[:, 0, :, :]
    
    chain_final_frame_0 = chain_batch_original_molecule[0, :, :]
    chain_final_frame_0_batch = chain_final_frame_0.repeat(keep_frames, 1, 1)
    original_linker_mask_batch = data["linker_mask"][0].squeeze().repeat(keep_frames, 1).cpu()

    original_positions = data["positions"][0]
    chain_positions = chain_batch_original_molecule[0, :, :3] 

    position_differences = original_positions - chain_positions
    position_differences = position_differences[data["fragment_mask"].squeeze().bool()][0]
    
    #add position differences to the chain_positions positions
    chain_final_frame_0_batch[:, :, :3] = chain_final_frame_0_batch[:, :, :3] + position_differences
    chain_batch_original_molecule[:, :, :3] = chain_batch_original_molecule[:, :, :3] + position_differences
    
    #compute Hausdorff distance between the frame 0 and the rest of the frames
    hausdorff_distances_original = compute_hausdorff_distance_batch(chain_final_frame_0_batch.cpu(), chain_batch_original_molecule.cpu(), mask1=original_linker_mask_batch, mask2=original_linker_mask_batch) 


    #save and visualize original chain

    for i in range(len(data['positions'])):
            chain = chain_batch_original[:, i, :, :]
            assert chain.shape[0] == keep_frames
            assert chain.shape[1] == data['positions'].shape[1]
            assert chain.shape[2] == data['positions'].shape[2] + data['one_hot'].shape[2] + model.include_charges

            # Saving chains
            name = str(data_index)

            mapping_output = os.path.join(mapping_output_dir, str(data_index), "original", "graphs")
            os.makedirs(mapping_output, exist_ok=True)
            
            #save initial random distrubution with noise
            positions_combined = torch.zeros_like(data['positions'])
            one_hot_combined = torch.zeros_like(data['one_hot'])

            # Iterate over each atom and decide whether to use original or noisy data
            for atom_idx in range(data['positions'].shape[1]):
                if data['fragment_mask'][0, atom_idx] == 1:
                    # Use original positions and features for fragment atoms
                    positions_combined[:, atom_idx, :] = data['positions'][:, atom_idx, :]
                    one_hot_combined[:, atom_idx, :] = data['one_hot'][:, atom_idx, :]
                else:
                    # Use noisy positions and features for linker atoms
                    positions_combined[:, atom_idx, :] = noisy_positions_present_atoms[:, atom_idx, :]
                    one_hot_combined[:, atom_idx, :] = noisy_features_present_atoms[:, atom_idx, :]
            
            save_xyz_file(
                mapping_output,
                one_hot_combined,
                positions_combined,
                node_mask_original[i].unsqueeze(0),
                names=[f'{name}_' + str(keep_frames)],
                is_geom=model.is_geom
            )

            
            one_hot = chain[:, :, 3:]
            positions = chain[:, :, :3]
            chain_node_mask = torch.cat([node_mask_original[i].unsqueeze(0) for _ in range(keep_frames)], dim=0)
            names = [f'{name}_{j}' for j in range(keep_frames + 1)]

            save_xyz_file(mapping_output, one_hot, positions, chain_node_mask, names=names, is_geom=model.is_geom)

            visualize_mapping_graph(
                    mapping_output,
                    spheres_3d=False,
                    alpha=1.0,
                    bg='white',
                    is_geom=model.is_geom,
                    fragment_mask=data['fragment_mask'][i].squeeze(),
                    phi_values=phi_values,
                    colormap='coolwarm_r' #reversed heatmap for distance-based importance
                )
            
            mapping_output_structure = os.path.join(mapping_output_dir, str(data_index), "original", "structures")
            os.makedirs(mapping_output, exist_ok=True)

            visualize_mapping_structure(
                    file_names=names,
                    generation_folder = mapping_output,
                    shapley_values = phi_values,
                    fragment_mask = data['fragment_mask'][0].cpu().numpy(),
                    linker_mask = data['linker_mask'][0].cpu().numpy(),
                    save_folder = mapping_output_structure,
                    colormap='coolwarm_r'
                )


    # Create a line plot for Hausdorff distances
    plt.figure(figsize=(10, 6))
    plt.gca().set_facecolor('white')
    #reverse hausdorff distances
    hausdorff_distances_original = hausdorff_distances_original[::-1]
    sns.lineplot(data=hausdorff_distances_original, marker='o')
    plt.title('Hausdorff Distance Trend')
    plt.xlabel('Frame')
    plt.ylabel('Hausdorff Distance')
    plt.xticks(ticks=range(keep_frames), labels=range(keep_frames-1, -1, -1))  # Show all 10 frames on the x-axis
    plt.ylim(bottom=0)  # Ensure the y-axis starts at 0
    #add white background


    SAVE_PATH = SAVE_FOLDER + "/" +  DATASET_NAME + "/seed_" + str(SEED) + "/" + str(data_index) + "/"


    os.makedirs(SAVE_PATH, exist_ok=True)

    plt.savefig(SAVE_PATH + "hausdorff_distance_trend_original.png", dpi = 300)
    plt.savefig(SAVE_PATH + "hausdorff_distance_trend_original.pdf", dpi = 300)
    
    plt.close()

    # Save hausdorff_distances_original to a file using pickle
    hausdorff_distances_original_path = os.path.join(SAVE_PATH, "hausdorff_distances_original.pkl")

    with open(hausdorff_distances_original_path, "wb") as f:
        pickle.dump(hausdorff_distances_original, f)
        
    fragment_mask = data["fragment_mask"].squeeze().bool()
    linker_mask = data["linker_mask"].squeeze().bool()
    phi_values_tensor = torch.tensor(phi_values)

    #get indices of phi_values_tensor from lower to higher
    sorted_phi_values, sorted_indices = torch.sort(phi_values_tensor)
    # reversed_indices = torch.flip(sorted_indices, [0])

    #take top k neighbors
    shapley_value_indices_keep = sorted_indices[top_k_neighs:]
    top_indices_to_perturb = sorted_indices[:top_k_neighs]
    sorted_phi_values = sorted_phi_values[:top_k_neighs]
    
    ##################################################
    #atom removal - removing top k neighbors
    ##################################################

    if not ATOM_INJECTION:
        data_temp = copy.deepcopy(data)

        noisy_positions_present_atoms_temp = noisy_positions_present_atoms.clone()
        noisy_features_present_atoms_temp = noisy_features_present_atoms.clone()
        
        #retrieve indices of fragment and linker atoms from atom_mask
        fragment_atoms_indices = torch.where(fragment_mask)[0]
        fragment_atoms_indices = fragment_atoms_indices.to(device)
        linker_atoms_indices = torch.where(linker_mask)[0]
        linker_atoms_indices = linker_atoms_indices.to(device)

        fragment_atoms_indices_keep = None
        
        
        #we remove all top k and make sure anchor atoms are kept
        fragment_atoms_indices_keep = torch.cat((shapley_value_indices_keep.to(device), torch.where(data_temp["anchors"].squeeze() == 1)[0].to(device)))
            
        
        #remove duplicates
        fragment_atoms_indices_keep = torch.unique(fragment_atoms_indices_keep)
            
        
        fragment_atoms_indices_keep_tensor = torch.Tensor(fragment_atoms_indices_keep).to(device)
        
        #keep only fragment_atoms_indices_keep and linker_atoms_indices
        atom_indices_to_keep = torch.cat((fragment_atoms_indices_keep_tensor, linker_atoms_indices)).to(device)

        #remove atoms from molecule
        data_temp["positions"] = data_temp["positions"][:, atom_indices_to_keep, :]
        data_temp["one_hot"] = data_temp["one_hot"][:, atom_indices_to_keep, :]
        data_temp["charges"] = data_temp["charges"][:, atom_indices_to_keep]
        data_temp["fragment_mask"] = data_temp["fragment_mask"][:, atom_indices_to_keep]
        data_temp["linker_mask"] = data_temp["linker_mask"][:, atom_indices_to_keep]
        data_temp["atom_mask"] = data_temp["atom_mask"][:, atom_indices_to_keep]
        data_temp["anchors"] = data_temp["anchors"][:, atom_indices_to_keep]
        edge_mask_to_keep = (data_temp["atom_mask"].unsqueeze(1) * data_temp["atom_mask"]).flatten()
        data_temp["edge_mask"] = edge_mask_to_keep

        #remove atoms from noisy features and positions
        noisy_positions_present_atoms_temp = noisy_positions_present_atoms_temp[:, atom_indices_to_keep, :]
        noisy_features_present_atoms_temp = noisy_features_present_atoms_temp[:, atom_indices_to_keep, :]

        phi_values_array = np.array(phi_values)
        cmap = plt.cm.get_cmap('coolwarm_r') #reversed heatmap for distance-based importance
        norm = plt.Normalize(vmin=min(phi_values_array), vmax=max(phi_values_array))
        colors_fragment_shadow_original = cmap(norm(phi_values_array))
        
        colors_fragment_shadow = cmap(norm(phi_values_array))
        #remove atoms from color array
        
        molecule_perturbation_original_positions = data_temp["positions"].clone()[0]

        
        colors_fragment_shadow = colors_fragment_shadow[fragment_atoms_indices_keep.cpu().numpy()]

        chain_batch, node_mask = model.sample_chain(data_temp, keep_frames=keep_frames, noisy_positions=noisy_positions_present_atoms_temp, noisy_features=noisy_features_present_atoms_temp)

        chain_batch_molecule_pertubation = chain_batch[:, 0, :, :]

        mask_to_use = data_temp["linker_mask"][0].squeeze().repeat(keep_frames, 1).cpu()

        chain_perturbation_positions = chain_batch_molecule_pertubation[0, :, :3]

        position_differences_perturb = molecule_perturbation_original_positions - chain_perturbation_positions
        position_differences_perturb = position_differences_perturb[data_temp["fragment_mask"].squeeze().bool()][0]

        chain_batch_molecule_pertubation[:, :, :3] = chain_batch_molecule_pertubation[:, :, :3] + position_differences_perturb

        hausdorff_distances_perturbation = compute_hausdorff_distance_batch(chain_final_frame_0_batch.cpu(), chain_batch_molecule_pertubation.cpu(), mask1=original_linker_mask_batch, mask2=mask_to_use) #the linker atoms are the same since those are the frames of a single molecule

        # Create a line plot for Hausdorff distances
        plt.figure(figsize=(10, 6))
        plt.gca().set_facecolor('white')
        
        hausdorff_distances_perturbation = hausdorff_distances_perturbation[::-1]
        sns.lineplot(data=hausdorff_distances_perturbation, marker='o')
        plt.title('Hausdorff Distance Trend')
        plt.xlabel('Frame')
        plt.ylabel('Hausdorff Distance')
        plt.xticks(ticks=range(keep_frames), labels=range(keep_frames-1, -1, -1))  
        plt.ylim(bottom=0)  
        

        os.makedirs(SAVE_PATH, exist_ok=True)

        plt.savefig(SAVE_PATH + f"hausdorff_distance_trend_removal.png", dpi = 300)
        plt.savefig(SAVE_PATH + f"hausdorff_distance_trend_removal.pdf", dpi = 300)
        
        plt.close()
        
        # Save hausdorff_distances_perturbation to a file using pickle
        hausdorff_distances_perturbation_path = os.path.join(SAVE_PATH, f"hausdorff_distances_perturbation_removal.pkl")
        with open(hausdorff_distances_perturbation_path, "wb") as f:
            pickle.dump(hausdorff_distances_perturbation, f)

        for i in range(len(data_temp['positions'])):
            chain = chain_batch[:, i, :, :]
            assert chain.shape[0] == keep_frames
            assert chain.shape[1] == data_temp['positions'].shape[1]
            assert chain.shape[2] == data_temp['positions'].shape[2] + data_temp['one_hot'].shape[2] + model.include_charges

            name = str(data_index)

            mapping_output = os.path.join(mapping_output_dir, str(data_index), "removal", "graphs")

            os.makedirs(mapping_output, exist_ok=True)

            #save initial random distrubution with noise
            positions_combined = torch.zeros_like(data_temp['positions'])
            one_hot_combined = torch.zeros_like(data_temp['one_hot'])

            # Iterate over each atom and decide whether to use original or noisy data
            for atom_idx in range(data_temp['positions'].shape[1]):
                if data_temp['fragment_mask'][0, atom_idx] == 1:
                    # Use original positions and features for fragment atoms
                    positions_combined[:, atom_idx, :] = data_temp['positions'][:, atom_idx, :]
                    one_hot_combined[:, atom_idx, :] = data_temp['one_hot'][:, atom_idx, :]
                    # atom_mask_combined[:, atom_idx] = data_temp['atom_mask'][:, atom_idx]
                else:
                    # Use noisy positions and features for linker atoms
                    positions_combined[:, atom_idx, :] = noisy_positions_present_atoms_temp[:, atom_idx, :]
                    one_hot_combined[:, atom_idx, :] = noisy_features_present_atoms_temp[:, atom_idx, :]

            #save initial distribution
                
            save_xyz_file(
                mapping_output,
                one_hot_combined,
                positions_combined,
                node_mask[i].unsqueeze(0),
                names=[f'{name}_' + str(keep_frames)],
                is_geom=model.is_geom
            )

            one_hot = chain[:, :, 3:]
            positions = chain[:, :, :3]
            chain_node_mask = torch.cat([node_mask[i].unsqueeze(0) for _ in range(keep_frames)], dim=0)
            names = [f'{name}_{j}' for j in range(keep_frames)]

            save_xyz_file(mapping_output, one_hot, positions, chain_node_mask, names=names, is_geom=model.is_geom)

            visualize_mapping_graph(
                mapping_output,
                # file_indices = range(chain_batch_molecule_pertubation_before_injection.shape[0]),
                spheres_3d=False,
                alpha=1.0,
                bg='white',
                is_geom=model.is_geom,
                fragment_mask=data_temp['fragment_mask'][i].squeeze(),
                #phi_values=phi_values,
                colormap='coolwarm_r', #reversed heatmap for distance-based importance
                colors_fragment_shadow = colors_fragment_shadow
            )

            mapping_output_structure = os.path.join(mapping_output_dir, str(data_index), "removal", "structures")
            os.makedirs(mapping_output_structure, exist_ok=True)

            visualize_mapping_structure(
                    file_names = names + [f'{name}_{keep_frames}'],
                    generation_folder = mapping_output,
                    shapley_values = phi_values,
                    fragment_mask = data_temp['fragment_mask'][0].cpu().numpy(),
                    linker_mask = data_temp['linker_mask'][0].cpu().numpy(),
                    save_folder = mapping_output_structure,
                    colormap='coolwarm_r'
                    # fragment_atoms_to_color = fragment_atoms_indices_keep.cpu().numpy() 
                )
    
    ##################################################
    #atom removal AND injection - removing top k neighbors
    ################################################## 
    else:    
        data_temp = copy.deepcopy(data)
        
        noisy_positions_present_atoms_temp = noisy_positions_present_atoms.clone()
        noisy_features_present_atoms_temp = noisy_features_present_atoms.clone()
        
        #retrieve indices of fragment and linker atoms from atom_mask
        fragment_atoms_indices = torch.where(fragment_mask)[0]
        fragment_atoms_indices = fragment_atoms_indices.to(device)
        linker_atoms_indices = torch.where(linker_mask)[0]
        linker_atoms_indices = linker_atoms_indices.to(device)
        

        fragment_atoms_indices_keep = None
        
        
        #we remove all top k and make sure anchor atoms are kept
        fragment_atoms_indices_keep = torch.cat((shapley_value_indices_keep.to(device), torch.where(data_temp["anchors"].squeeze() == 1)[0].to(device)))
            
        
        #remove duplicates
        fragment_atoms_indices_keep = torch.unique(fragment_atoms_indices_keep)
            
        
        fragment_atoms_indices_keep_tensor = torch.Tensor(fragment_atoms_indices_keep).to(device)
        
        #keep only fragment_atoms_indices_keep and linker_atoms_indices
        atom_indices_to_keep = torch.cat((fragment_atoms_indices_keep_tensor, linker_atoms_indices)).to(device)

        #remove atoms from molecule
        data_temp["positions"] = data_temp["positions"][:, atom_indices_to_keep, :]
        data_temp["one_hot"] = data_temp["one_hot"][:, atom_indices_to_keep, :]
        data_temp["charges"] = data_temp["charges"][:, atom_indices_to_keep]
        data_temp["fragment_mask"] = data_temp["fragment_mask"][:, atom_indices_to_keep]
        data_temp["linker_mask"] = data_temp["linker_mask"][:, atom_indices_to_keep]
        data_temp["atom_mask"] = data_temp["atom_mask"][:, atom_indices_to_keep]
        data_temp["anchors"] = data_temp["anchors"][:, atom_indices_to_keep]
        edge_mask_to_keep = (data_temp["atom_mask"].unsqueeze(1) * data_temp["atom_mask"]).flatten()
        data_temp["edge_mask"] = edge_mask_to_keep

        #remove atoms from noisy features and positions
        noisy_positions_present_atoms_temp = noisy_positions_present_atoms_temp[:, atom_indices_to_keep, :]
        noisy_features_present_atoms_temp = noisy_features_present_atoms_temp[:, atom_indices_to_keep, :]

        phi_values_array = np.array(phi_values)
        cmap = plt.cm.get_cmap('coolwarm_r') #reversed heatmap for distance-based importance
        norm = plt.Normalize(vmin=min(phi_values_array), vmax=max(phi_values_array))
        colors_fragment_shadow_original = cmap(norm(phi_values_array))
        
        colors_fragment_shadow = cmap(norm(phi_values_array))
        #remove atoms from color array
        
        molecule_perturbation_original_positions = data_temp["positions"].clone()[0]

        
        colors_fragment_shadow = colors_fragment_shadow[fragment_atoms_indices_keep.cpu().numpy()]
            
        chain_before_injection, node_mask_before_injection, chain_after_injection, node_mask_after_injection = model.sample_chain_atom_injection(data_temp, keep_frames=keep_frames, noisy_positions=noisy_positions_present_atoms_temp, noisy_features=noisy_features_present_atoms_temp, orginal_data = data, noisy_positions_original = noisy_positions_present_atoms, noisy_features_original = noisy_features_present_atoms, atom_indices_kept = atom_indices_to_keep, injection_step = injection_step)

            
        chain_batch_molecule_pertubation_before_injection = chain_before_injection[:, 0, :, :]
        chain_batch_molecule_pertubation_after_injection = chain_after_injection[:, 0, :, :]

        # Get the indices of tensors with all zero elements for the two chains
        # Remove tensors with all zero elements from chain_batch_molecule_pertubation_before_injection
        non_zero_indices_before_injection = torch.any(chain_batch_molecule_pertubation_before_injection != 0, dim=(1, 2))
        chain_batch_molecule_pertubation_before_injection = chain_batch_molecule_pertubation_before_injection[non_zero_indices_before_injection]

        # Remove tensors with all zero elements from chain_batch_molecule_pertubation_after_injection
        non_zero_indices_after_injection = torch.any(chain_batch_molecule_pertubation_after_injection != 0, dim=(1, 2))
        chain_batch_molecule_pertubation_after_injection = chain_batch_molecule_pertubation_after_injection[non_zero_indices_after_injection]


        mask_to_use = None
        
        mask_to_use_before_injection = data_temp["linker_mask"][0].squeeze().repeat(keep_frames, 1).cpu()
        mask_to_use_after_injection = original_linker_mask_batch
        
        chain_perturbation_positions_before_injection = chain_batch_molecule_pertubation_before_injection[0, :, :3]  # Assuming the first 3 columns are the positions
        chain_perturbation_positions_after_injection = chain_batch_molecule_pertubation_after_injection[0, :, :3]  # Assuming the first 3 columns are the positions

        position_differences_perturb_before_injection = molecule_perturbation_original_positions - chain_perturbation_positions_before_injection
        position_differences_perturb_before_injection = position_differences_perturb_before_injection[data_temp["fragment_mask"].squeeze().bool()][0]

        position_differences_perturb_after_injection = original_positions - chain_perturbation_positions_after_injection
        position_differences_perturb_after_injection = position_differences_perturb_after_injection[data["fragment_mask"].squeeze().bool()][0]

        chain_batch_molecule_pertubation_before_injection[:, :, :3] = chain_batch_molecule_pertubation_before_injection[:, :, :3] + position_differences_perturb_before_injection
        chain_batch_molecule_pertubation_after_injection[:, :, :3] = chain_batch_molecule_pertubation_after_injection[:, :, :3] + position_differences_perturb_after_injection

        chain_final_frame_0_batch_before_injection = chain_final_frame_0.repeat(chain_batch_molecule_pertubation_before_injection.shape[0], 1, 1)
        
        hausdorff_distances_perturbation_before_injection = compute_hausdorff_distance_batch(chain_final_frame_0_batch_before_injection.cpu(), chain_batch_molecule_pertubation_before_injection.cpu(), mask1=original_linker_mask_batch, mask2=mask_to_use_before_injection) 

        chain_final_frame_0_batch_after_injection = chain_final_frame_0.repeat(chain_batch_molecule_pertubation_after_injection.shape[0], 1, 1)
        hausdorff_distances_perturbation_after_injection = compute_hausdorff_distance_batch(chain_final_frame_0_batch_after_injection.cpu(), chain_batch_molecule_pertubation_after_injection.cpu(), mask1=original_linker_mask_batch, mask2=mask_to_use_after_injection) 

        hausdorff_distances_perturbation = hausdorff_distances_perturbation_after_injection + hausdorff_distances_perturbation_before_injection

        
        # Create a line plot for Hausdorff distances
        plt.figure(figsize=(10, 6))
        plt.gca().set_facecolor('white')
        
        hausdorff_distances_perturbation = hausdorff_distances_perturbation[::-1]
        sns.lineplot(data=hausdorff_distances_perturbation, marker='o')
        plt.title('Hausdorff Distance Trend')
        plt.xlabel('Frame')
        plt.ylabel('Hausdorff Distance')
        plt.xticks(ticks=range(keep_frames), labels=range(keep_frames-1, -1, -1))  
        plt.ylim(bottom=0)  
        

        os.makedirs(SAVE_PATH, exist_ok=True)

        plt.savefig(SAVE_PATH + f"hausdorff_distance_trend_injection.png", dpi = 300)
        plt.savefig(SAVE_PATH + f"hausdorff_distance_trend_injection.pdf", dpi = 300)
        
        plt.close()
        
        # Save hausdorff_distances_perturbation to a file using pickle
        hausdorff_distances_perturbation_path = os.path.join(SAVE_PATH, f"hausdorff_distances_perturbation_injection.pkl")
        with open(hausdorff_distances_perturbation_path, "wb") as f:
            pickle.dump(hausdorff_distances_perturbation, f)


        for i in range(len(data_temp['positions'])):
            
            assert chain_batch_molecule_pertubation_before_injection.shape[1] == data_temp['positions'].shape[1]
            assert chain_batch_molecule_pertubation_before_injection.shape[2] == data_temp['positions'].shape[2] + data_temp['one_hot'].shape[2] + model.include_charges

            
            name = str(data_index)

            mapping_output = os.path.join(mapping_output_dir, str(data_index), "injection", "graphs")

            os.makedirs(mapping_output, exist_ok=True)
            
            #save initial random distrubution with noise
            positions_combined = torch.zeros_like(data_temp['positions'])
            one_hot_combined = torch.zeros_like(data_temp['one_hot'])

            # Iterate over each atom and decide whether to use original or noisy data
            for atom_idx in range(data_temp['positions'].shape[1]):
                if data_temp['fragment_mask'][0, atom_idx] == 1:
                    # Use original positions and features for fragment atoms
                    positions_combined[:, atom_idx, :] = data_temp['positions'][:, atom_idx, :]
                    one_hot_combined[:, atom_idx, :] = data_temp['one_hot'][:, atom_idx, :]
                else:
                    # Use noisy positions and features for linker atoms
                    positions_combined[:, atom_idx, :] = noisy_positions_present_atoms_temp[:, atom_idx, :]
                    one_hot_combined[:, atom_idx, :] = noisy_features_present_atoms_temp[:, atom_idx, :]

            #save initial distribution

            save_xyz_file(
                mapping_output,
                one_hot_combined,
                positions_combined,
                node_mask_before_injection[i].unsqueeze(0),
                names=[f'{name}_' + str(keep_frames)],
                is_geom=model.is_geom
            )

            
            one_hot_before_injection = chain_batch_molecule_pertubation_before_injection[:, :, 3:]
            positions_before_injection = chain_batch_molecule_pertubation_before_injection[:, :, :3]
            chain_node_mask_before_injection = torch.cat([node_mask_before_injection[i].unsqueeze(0) for _ in range(chain_batch_molecule_pertubation_before_injection.shape[0])], dim=0)
            names_before_injection = [f'{name}_{j}' for j in range(chain_batch_molecule_pertubation_after_injection.shape[0],keep_frames)]

            save_xyz_file(mapping_output, one_hot_before_injection, positions_before_injection, chain_node_mask_before_injection, names=names_before_injection, is_geom=model.is_geom)

            
            # draw_atom_indices = (fragment_atoms_indices_keep.tolist(), linker_atoms_indices.tolist())
            

            visualize_mapping_graph(
                    mapping_output,
                    # file_indices = range(chain_batch_molecule_pertubation_before_injection.shape[0]),
                    spheres_3d=False,
                    alpha=1.0,
                    bg='white',
                    is_geom=model.is_geom,
                    fragment_mask=data_temp['fragment_mask'][i].squeeze(),
                    #phi_values=phi_values,
                    colormap='coolwarm_r', #reversed heatmap for distance-based importance
                    colors_fragment_shadow = colors_fragment_shadow
                )

            mapping_output_structure = os.path.join(mapping_output_dir, str(data_index), "injection", "structures")
            os.makedirs(mapping_output_structure, exist_ok=True)

            visualize_mapping_structure(
                    file_names=names_before_injection + [f'{name}_{keep_frames}'],
                    generation_folder = mapping_output,
                    shapley_values = phi_values,
                    fragment_mask = data_temp['fragment_mask'][0].cpu().numpy(),
                    linker_mask = data_temp['linker_mask'][0].cpu().numpy(),
                    save_folder = mapping_output_structure,
                    colormap='coolwarm_r'
                    # fragment_atoms_to_color = fragment_atoms_indices_keep.cpu().numpy() 
                )

        
        #save for after injection
        for i in range(len(data["positions"])):


            assert chain_batch_molecule_pertubation_after_injection.shape[1] == data["positions"].shape[1]
            assert chain_batch_molecule_pertubation_after_injection.shape[2] == data["positions"].shape[2] + data["one_hot"].shape[2] + model.include_charges

            # Saving chains
            name = str(data_index)
            mapping_output = os.path.join(mapping_output_dir, str(data_index), "injection", "graphs")
            os.makedirs(mapping_output, exist_ok=True)
            
            
            #save initial random distrubution with noise
            positions_combined = torch.zeros_like(data['positions'])
            one_hot_combined = torch.zeros_like(data['one_hot'])

            # Iterate over each atom and decide whether to use original or noisy data
            for atom_idx in range(data['positions'].shape[1]):
                if data['fragment_mask'][0, atom_idx] == 1:
                    # Use original positions and features for fragment atoms
                    positions_combined[:, atom_idx, :] = data['positions'][:, atom_idx, :]
                    one_hot_combined[:, atom_idx, :] = data['one_hot'][:, atom_idx, :]
                else:
                    # Use noisy positions and features for linker atoms
                    positions_combined[:, atom_idx, :] = noisy_positions_present_atoms[:, atom_idx, :]
                    one_hot_combined[:, atom_idx, :] = noisy_features_present_atoms[:, atom_idx, :]

            
            one_hot_after_injection = chain_batch_molecule_pertubation_after_injection[:, :, 3:]
            positions_after_injection = chain_batch_molecule_pertubation_after_injection[:, :, :3]
            chain_node_mask_after_injection = torch.cat([node_mask_after_injection[i].unsqueeze(0) for _ in range(chain_batch_molecule_pertubation_after_injection.shape[0])], dim=0)
            names_after_injection = [f'{name}_{j}' for j in range(chain_batch_molecule_pertubation_after_injection.shape[0])]

            # print("Names after injection: ", names_after_injection)

            save_xyz_file(mapping_output, one_hot_after_injection, positions_after_injection, chain_node_mask_after_injection, names=names_after_injection, is_geom=model.is_geom)


            visualize_mapping_graph(
                    mapping_output,
                    file_indices = range(chain_batch_molecule_pertubation_after_injection.shape[0]),
                    spheres_3d=False,
                    alpha=1.0,
                    bg='white',
                    is_geom=model.is_geom,
                    fragment_mask=data['fragment_mask'][i].squeeze(),
                    #phi_values=phi_values,
                    colormap='coolwarm_r', #reversed heatmap for distance-based importance
                    colors_fragment_shadow = colors_fragment_shadow_original
                )
            
            mapping_output_structure = os.path.join(mapping_output_dir, str(data_index), "injection", "structures")
            os.makedirs(mapping_output_structure, exist_ok=True)

            visualize_mapping_structure(
                    file_names=names_after_injection,
                    generation_folder = mapping_output,
                    shapley_values = phi_values,
                    fragment_mask = data['fragment_mask'][0].cpu().numpy(),
                    linker_mask = data['linker_mask'][0].cpu().numpy(),
                    save_folder = mapping_output_structure,
                    colormap='coolwarm_r' 
                )


    del data_temp
    del noisy_features_present_atoms_temp
    del noisy_positions_present_atoms_temp
    start += len(data['positions'])

  0%|          | 0/30 [00:00<?, ?it/s]

[10:44:18] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[10:44:18] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[10:44:46] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[10:45:13] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[10:45:23] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[10:45:23] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[10:45:42] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[10:45:42] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[10:45:42] WARNING: could not find number of expected rings. Switching to an approximate ring fi